> 조별활동 시 여기 내용대로 최적의 파라미터를 찾아서 해야 함.

#### 교차검증과 최적의 파라미터 찾기
 : Train 과 Test 로 나눈 것 중, Train을 Train 과 Valid 로 나누고 여러 모델을 시행해보면서 교차 검증하고, 최적의 파라미터를 찾아야 함. 제일 좋은 모델을 찾으면 그거로 Test 시험.
- 머신 러닝을 사용할 때 모델의 정확도를 측정하기 위해 반드시 사용해야 하는 방법
- 딥러닝 시에는 데이터의 크기가 크므로 이 방법은 사용할 필요가 없다. (데이터 수가 많아서 뭘 집어도 잘 나옴)

In [1]:
import pandas as pd
wine = pd.read_csv('../Data/wine.csv')
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


#### Feature 와 Target 분리

In [ ]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

----
#### 검증 세트 추가
- 19번 파일에서 훈련세트와 테스트세트만 가지고 작업을 하였지만 테스트 세트 작업에 파라미터를 조절하여 정확성을 높히면 실전 사용 시 문제가 발생한다.
- 이런 과정을 방지하기 위해 훈련세트, 검증세트, 테스트세트로 구분하여 분석 작업을 한다.

In [ ]:
# 전체 세트 중 훈련세트와 테스트세트를 8:2 기준으로 분리  (Train / Test)
from sklearn.model_selection import train_test_split

In [4]:
train_input, test_input, train_target, test_target = \
    train_test_split(
        data,
        target,
        test_size=0.2,
        random_state=42
    )

In [6]:
# 훈련세트 중 훈련세트와 검증세트를 8:2 기준으로 분리  (Train / Valid)
sub_input, val_input, sub_target, val_target = \
    train_test_split(
        train_input,
        train_target,
        test_size=0.2,
        random_state=42
    )

In [7]:
# 훈련세트, 검증세트, 테스트세트의 크기 구하기
print("Train : ", sub_input.shape)
print("Valid : ", val_input.shape)
print("Test : ", test_input.shape)

Train :  (4157, 3)
Valid :  (1040, 3)
Test :  (1300, 3)


In [ ]:
# 훈련세트와 검증세트로 결정트리 모델 만들기
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    min_impurity_decrease=0.0005  # 2. 이걸 넣어줘서 최적의 정확도를 봄
)
dt.fit(sub_input, sub_target)

print("Train : ", dt.score(sub_input, sub_target))
print("Valid : ", dt.score(val_input, val_target)) # 1. 여기까지 일단 보고 수치를 확인한 뒤에 부족하면

Train :  0.8946355544864084
Valid :  0.8653846153846154


In [11]:
# Test Set 로 최종 확인
print("Test : ", dt.score(test_input, test_target))  # test 의 점수가 valid 와 비슷하게 나오면 성공인 거임.

Test :  0.8638461538461538


----
#### 교차검증(Cross Validation)
- 교차검증의 한 파트를 폴드라고 하며 교차검증의 기본 Fold 는 5 이다.
- 훈련세트와 검증세트를 바꾸어 가며 정확도를 구하는 방법이다.
- 전체에 대한 정확도는 해당 값들의 평균으로 구한다.

In [13]:
from sklearn.model_selection import cross_validate
scores = cross_validate(dt, train_input, train_target)
scores     # 5가 기본값이므로 5가지의 score 가 나옴

{'fit_time': array([0.00373936, 0.00327039, 0.00334001, 0.00417304, 0.00325465]),
 'score_time': array([0.00079346, 0.00081372, 0.00075388, 0.00085855, 0.00070381]),
 'test_score': array([0.86538462, 0.86923077, 0.8825794 , 0.84985563, 0.87102984])}

In [14]:
scores['test_score'].mean()

np.float64(0.8676160509365515)

----
#### Optuna
- 최적화 알고리즘 기반 탐색
- 정밀하고 효율적인 Hyper Parameter 탐색

In [ ]:
# !pip install optuna

In [18]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score   # Classifier 면 cross_val_score , 아니면 cross_validate
from sklearn.datasets import load_iris

In [19]:
iris = load_iris()
train_input, test_input, train_target, test_target = \
    train_test_split(
        iris.data,
        iris.target,
        test_size=0.2,
        random_state=42,
        stratify=iris.target
    )

- 랜덤포레스트의 경우 아래 양식을 따름. (다른 것들의 양식은 GPT 물어보면 친절히 써줌. 특히, 뒤에 숫자 범위 모를거임) (EX_Optuna 로 ~~ 의 함수식을 써줘)

In [22]:
def find_param(trial):
    # 탐색할 하이퍼파라미터 정의
    n_estimators = trial.suggest_int('n_estimator', 50, 200)    # 50 ~ 200 까지 트라이 해봐서 좋은 점수 내봐라.
    max_depth = trial.suggest_int('max_depth', 2, 20)      # 2 ~ 20 까지 max_depth 가 뭐가 좋을지 찾아봐라.
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)    # leaf(마지막)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])      # 뽑힌 데이터를 다시 뽑게 할지 말지. 분류니까 int 형식이 아니라 categorical
    clf = RandomForestClassifier(     # 위에 최적의 점수가 나올 걸 찾고 clf에 넣음
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        random_state=42
    )
    return cross_val_score(clf, train_input, train_target, cv=5).mean()

- 뽑아온 데이터를 다시 집어넣어서 뽑게 함 : boot strap

In [ ]:
study = optuna.create_study(direction='maximize')  # 위의 점수 중 제일 좋은 값
study.optimize(find_param, n_trials=50)    # 50번 시도
print("최적의 파라미터 : ", study.best_params)

[I 2026-07-03 14:07:24,834] A new study created in memory with name: no-name-96344425-d859-4df4-ae6f-7942c8aa2ca2
[I 2026-07-03 14:07:25,350] Trial 0 finished with value: 0.9583333333333334 and parameters: {'n_estimator': 127, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 2, 'bootstrap': False}. Best is trial 0 with value: 0.9583333333333334.
[I 2026-07-03 14:07:25,891] Trial 1 finished with value: 0.95 and parameters: {'n_estimator': 102, 'max_depth': 9, 'min_samples_split': 3, 'min_samples_leaf': 4, 'bootstrap': True}. Best is trial 0 with value: 0.9583333333333334.
[I 2026-07-03 14:07:26,885] Trial 2 finished with value: 0.9416666666666668 and parameters: {'n_estimator': 188, 'max_depth': 17, 'min_samples_split': 10, 'min_samples_leaf': 4, 'bootstrap': True}. Best is trial 0 with value: 0.9583333333333334.
[I 2026-07-03 14:07:27,117] Trial 3 finished with value: 0.95 and parameters: {'n_estimator': 57, 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 4, '

최적의 파라미터 :  {'n_estimator': 127, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 2, 'bootstrap': False}


##### 위에 최적의 파라미터 결과가 나옴
{'n_estimator': 127, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 2, 'bootstrap': False}

In [24]:
# 최종 모델 Hyper Parameter 조정하기
# 랜덤 포레스트 분류기
rf = RandomForestClassifier(
    n_estimators=127,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=2,
    bootstrap=False,
    random_state=42
    )

In [26]:
rf.fit(train_input, train_target)

,n_estimators,127
,criterion,'gini'
,max_depth,10
,min_samples_split,10
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [27]:
print("Train : ", rf.score(train_input, train_target))
print("Test : ", rf.score(test_input, test_target))

Train :  0.9833333333333333
Test :  0.9666666666666667
